In [ ]:
# Do use
# Block-100
!pip install -q torch transformers accelerate bitsandbytes transformers-utils torch

In [ ]:
# Do use
# Block-101

import sys
import transformers.utils
from typing import TypedDict

# Re-create the missing LossKwargs structural type that v5 dropped
if not hasattr(transformers.utils, "LossKwargs"):
    class LossKwargs(TypedDict, total=False):
        num_items_in_batch: int

    transformers.utils.LossKwargs = LossKwargs

In [ ]:
# Do use
# Block-102

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "microsoft/Phi-4-mini-instruct"

print("📥 Loading Tokenizer natively...")
tokenizer = AutoTokenizer.from_pretrained(model_id) # trust_remote_code removed

print("📥 Loading Model weights natively via bitsandbytes...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load the model directly using native Transformers architecture
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
    # trust_remote_code=True REMOVED to prevent config parsing crashes
)

print("✅ Model and Tokenizer loaded successfully without conflicts!")


📥 Loading Tokenizer natively...
📥 Loading Model weights natively via bitsandbytes...


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

✅ Model and Tokenizer loaded successfully without conflicts!


In [ ]:
# Do not use
# Block-103

# =====================================================================
# Step 3: Run the Targeted Code Review Script (Block-602)
# =====================================================================

chat_messages = [
    {
        "role": "system",
        "content": "You are an expert Python code reviewer. Identify critical runtime bugs, styling issues, and suggest robust corrections."
    },
    {
        "role": "user",
        "content": "Perform a concise code review on this function:\n```python\ndef process(data):\n    x = data/0\n    return x\n```"
    }
]

print("\n🤖 Running strict instruction-based code review...")

# Process the chat history using the model's template.
# return_dict=True ensures it returns a dictionary structure instead of a raw list,
# preventing the 'list object has no attribute keys' crash during **inputs unpacking.
inputs = tokenizer.apply_chat_template(
    chat_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# Generate tokens deterministically (temperature=0.1) for debugging accuracy
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.1
)

# Decode only the newly generated response tokens by slicing off the input prompt
prompt_length = inputs["input_ids"].shape[-1]
review_response = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

print("\n=== Phi-4-mini's High-Quality Code Review ===")
print(review_response)

In [ ]:
# Do use
# Block-104

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain quantum computing in one short paragraph."}
]

# Format using the model's official chat template
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt")

# Generate response
outputs = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

print(response)


Quantum computing leverages the principles of quantum mechanics, utilizing quantum bits or qubits, which unlike classical bits can exist in multiple states simultaneously through superposition. This allows quantum computers to process vast amounts of data and perform complex calculations at unprecedented speeds, potentially revolutionizing fields such as cryptography, optimization, and drug discovery.


In [ ]:
# use now
# Block-602
# 3. Define the POC targeted Code Review Script
# 1. Format using the strict Chat ML layout to force an 'instruction' context
chat_messages = [
    {"role": "system", "content": "You are an expert Python code reviewer. Identify critical runtime bugs, styling issues, and suggest robust corrections."},
    {"role": "user", "content": f"Perform a concise code review on this function:\n```python\ndef process(data):\n    x = data/0\n    return x\n```"}
]

# 2. Convert standard message layout into a raw prompt string
# llama-cpp-python automatically handles chat conversions behind the scenes:
print("\n🤖 Running strict instruction-based code review...")
response = model.create_chat_completion(
    messages=chat_messages,
    max_tokens=512,
    temperature=0.1 # Dropped to 0.1 for sharp, exact debugging accuracy
)

print("\n=== Nemotron's High-Quality Code Review ===")
print(response["choices"][0]["message"]["content"])

In [ ]:
# Do use
# Block-200

!pip install llama-cpp-python

In [ ]:
# Do use
# Block-201

!pip install huggingface_hub

In [ ]:
# Do use
# Block-202

from llama_cpp import Llama

# This will automatically check Hugging Face, download the model if missing, and load it into memory
print("Loading model (will download first if not found localy)...")
llm = Llama.from_pretrained(
    repo_id="bartowski/Llama-3.2-1B-Instruct-GGUF",
    filename="Llama-3.2-1B-Instruct-Q4_K_M.gguf",
    n_ctx=2048,
    n_gpu_layers=0  # Set to 0 if you are using CPU only
)

In [ ]:
#Do not Use
#Block  203
from llama_cpp import Llama

# 1. Initialize the model
# Set n_gpu_layers to -1 to offload all layers to your GPU if using CUDA/Metal. Set to 0 for CPU only.
llm = Llama(
    model_path="Llama-3.2-1B-Instruct-Q4_K_M.gguf",
    n_ctx=2048,           # The context window length (supports up to 131k, but 2048/4096 is standard for local memory)
    n_threads=4,          # Adjust to your CPU physical core count
    n_gpu_layers=-1       # Offload to GPU (-1 for all layers, 0 for CPU)
)



In [ ]:
#Do  Use
#Block  204
# 2. Define your conversation history using the standard chat completion format
chat_messages = [
    {"role": "system", "content": "You are an expert Python code reviewer. Identify critical runtime bugs, styling issues, and suggest robust corrections."},
    {"role": "user", "content": f"Perform a concise code review on this function:\n```python\ndef process(data):\n    x = data/0\n    return x\n```"}
]

# 3. Generate the response
print("Generating response...")
response = llm.create_chat_completion(
    messages=chat_messages,
    max_tokens=256,      # Maximum number of tokens to generate
    temperature=0.7,     # Controls the randomness of the output
    top_p=0.9
)

# 4. Extract and print the assistant's reply
assistant_reply = response["choices"][0]["message"]["content"]
print("\nAssistant:")
print(assistant_reply)

In [ ]:
#Do  Use
#Block  301
!pip install "llama-cpp-python[server]"

In [ ]:
#Do  Use
#Block  302
python -m llama_cpp.server --model ./Llama-3.2-1B-Instruct-Q4_K_M.gguf --host 0.0.0.0 --port 8000 --n_ctx 2048 --n_gpu_layers -1


In [ ]:
#Do  Use
#Block  303
from openai import OpenAI

# Initialize client pointing to the host machine's IP address
client = OpenAI(
    base_url="http://<YOUR_HOST_IP_ADDRESS>:8000/v1",
    api_key="not-needed" # The local server doesn't require a key by default
)

completion = client.chat.completions.create(
    model="local-model",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a 3-word slogan for an eco-friendly brand."}
    ],
    temperature=0.7
)

print(completion.choices[0].message.content)

In [1]:
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 35.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]
